In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [2]:
df = pd.read_csv('BTC-USD.csv')

In [3]:
df.tail(10)

,Date,Open,High,Low,Close,Adj Close,Volume
3510,2024-04-27,63750.988281,63898.363281,62424.718750,63419.140625,63419.140625,19530783039
3511,2024-04-28,63423.515625,64321.484375,62793.597656,63113.230469,63113.230469,17334827993
3512,2024-04-29,63106.363281,64174.878906,61795.457031,63841.121094,63841.121094,26635912073
3513,2024-04-30,63839.417969,64703.332031,59120.066406,60636.855469,60636.855469,37840840057
3514,2024-05-01,60609.496094,60780.500000,56555.292969,58254.011719,58254.011719,48439780271
3515,2024-05-02,58253.703125,59602.296875,56937.203125,59123.433594,59123.433594,32711813559
3516,2024-05-03,59122.300781,63320.503906,58848.312500,62889.835938,62889.835938,33172023048
3517,2024-05-04,62891.031250,64494.957031,62599.351563,63891.472656,63891.472656,20620477992
3518,2024-05-05,63892.453125,64610.890625,62955.304688,64031.132813,64031.132813,18296164805
3519,2024-05-06,64032.632813,65425.679688,63752.597656,64601.843750,64601.843750,21053020160


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3520 entries, 0 to 3519
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       3520 non-null   str    
 1   Open       3520 non-null   float64
 2   High       3520 non-null   float64
 3   Low        3520 non-null   float64
 4   Close      3520 non-null   float64
 5   Adj Close  3520 non-null   float64
 6   Volume     3520 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 227.0 KB


In [5]:
df.isnull().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Data Preparation Section

In [6]:
df.drop(columns=['Adj Close'], inplace= True)

In [7]:
df.sort_index(inplace=True)

In [8]:
df.head(10)

,Date,Open,High,Low,Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600
6,2014-09-23,402.092010,441.557007,396.196991,435.790985,45099500
7,2014-09-24,435.751007,436.112000,421.131989,423.204987,30627700
8,2014-09-25,423.156006,423.519989,409.467987,411.574005,26814400
9,2014-09-26,411.428986,414.937988,400.009003,404.424988,21460800


### Stationarity checking of the dataset

In [10]:
from statsmodels.tsa.stattools import adfuller

In [11]:
def run_adf_test(Series):
    result = adfuller(Series.dropna())

    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'P value : {result[1]:.4f}')

    if result[1] <= 0.05:
       print("Result: Stationary (Reject H0)")
    else:
        print("Result: Non-Stationary (Fail to reject H0)")

print("--- Raw BTC Prices ---")
run_adf_test(df['Close'])


--- Raw BTC Prices ---
ADF Statistic : -0.8982
P value : 0.7886
Result: Non-Stationary (Fail to reject H0)


In [12]:
print("\n--- Log Returns ---")
log_returns = np.log(df['Close'] / df['Close'].shift(1))
run_adf_test(log_returns)


--- Log Returns ---
ADF Statistic : -60.5543
P value : 0.0000
Result: Stationary (Reject H0)


In [13]:
df['Log_Returns'] = log_returns.dropna()

In [14]:
df.head()

,Date,Open,High,Low,Close,Volume,Log_Returns
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800,NaN
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,-0.074643
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,-0.072402
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600,0.035111
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100,-0.024968


In [15]:
# Create a series that only contains the valid numbers
clean_log_returns = df['Log_Returns'].dropna()

# Now the ADF test will work perfectly
result = adfuller(clean_log_returns)

In [16]:
df = df.dropna()

In [17]:
df.head()

,Date,Open,High,Low,Close,Volume,Log_Returns
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,-0.074643
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,-0.072402
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600,0.035111
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100,-0.024968
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600,0.008317


### Outlier checking

In [18]:
mean = df['Log_Returns'].mean()
std = df['Log_Returns'].std()

# Define outliers as anything 3 standard deviations away
df.loc[:, 'Is_Outlier'] = (df['Log_Returns'] > mean + 3*std) | (df['Log_Returns'] < mean - 3*std)

# See how many outliers you found
print(df['Is_Outlier'].sum())

57


In [19]:
df.head()

,Date,Open,High,Low,Close,Volume,Log_Returns,Is_Outlier
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,-0.074643,False
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,-0.072402,False
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600,0.035111,False
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100,-0.024968,False
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600,0.008317,False


In [20]:
def create_features(df):

    # Existing features
    for lag in [1, 2, 3]:
        df[f'Log_Returns_Lag_{lag}'] = df['Log_Returns'].shift(lag)
        df[f'Volume_Lag_{lag}'] = df['Volume'].shift(lag)

    df['Volatility_7d'] = (
        df['Log_Returns']
        .rolling(7)
        .std()
    )

    return df

In [25]:
create_features(df) # creates Lag features and volatility

,Date,Open,High,Low,Close,Volume,Log_Returns,Is_Outlier,SMA_10,SMA_20,...,Momentum_5,Momentum_10,Momentum_20,Log_Returns_Lag_1,Volume_Lag_1,Log_Returns_Lag_2,Volume_Lag_2,Log_Returns_Lag_3,Volume_Lag_3,Volatility_7d
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,-0.074643,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,-0.072402,False,NaN,NaN,...,NaN,NaN,NaN,-0.074643,3.448320e+07,NaN,NaN,NaN,NaN,NaN
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600,0.035111,False,NaN,NaN,...,NaN,NaN,NaN,-0.072402,3.791970e+07,-0.074643,3.448320e+07,NaN,NaN,NaN
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100,-0.024968,False,NaN,NaN,...,NaN,NaN,NaN,0.035111,3.686360e+07,-0.072402,3.791970e+07,-0.074643,3.448320e+07,NaN
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600,0.008317,False,NaN,NaN,...,NaN,NaN,NaN,-0.024968,2.658010e+07,0.035111,3.686360e+07,-0.072402,3.791970e+07,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3515,2024-05-02,58253.703125,59602.296875,56937.203125,59123.433594,32711813559,0.014814,False,62730.899219,63474.952344,...,-0.067735,-0.115418,-0.120133,-0.040090,4.843978e+10,-0.051495,3.784084e+10,0.011467,2.663591e+10,0.024881
3516,2024-05-03,59122.300781,63320.503906,58848.312500,62889.835938,33172023048,0.061757,False,62379.155469,63428.370508,...,-0.003540,-0.052968,-0.014598,0.014814,3.271181e+10,-0.040090,4.843978e+10,-0.051495,3.784084e+10,0.037524
3517,2024-05-04,62891.031250,64494.957031,62599.351563,63891.472656,20620477992,0.015801,False,62340.612891,63336.007813,...,0.000789,-0.005996,-0.028100,0.061757,3.317202e+10,0.014814,3.271181e+10,-0.040090,4.843978e+10,0.038055
3518,2024-05-05,63892.453125,64610.890625,62955.304688,64031.132813,18296164805,0.002184,False,62295.555469,63366.253906,...,0.055977,-0.006988,0.009537,0.015801,2.062048e+10,0.061757,3.317202e+10,0.014814,3.271181e+10,0.037966


In [ ]:
# Moving averages
# Calculates the Simple Moving Average (SMA) over the last 10, 20, and 50 data periods.
df['SMA_10'] = df['Close'].rolling(10).mean()
df['SMA_20'] = df['Close'].rolling(20).mean()
df['SMA_50'] = df['Close'].rolling(50).mean()

# Distance from moving averages
# Divides the current closing price by the respective moving average and subtracts 1 to turn 
# the result into a positive or negative percentage difference.
df['Price_vs_SMA10'] = (
    df['Close'] / df['SMA_10']
) - 1

df['Price_vs_SMA20'] = (
    df['Close'] / df['SMA_20']
) - 1

df['Price_vs_SMA50'] = (
    df['Close'] / df['SMA_50']
) - 1

Above code gives details about the price which is:
Above trend?
Below trend?
Strong trend?
Weak trend?

This below code calculates percentage-based momentum for Bitcoin over 5, 10, and 20 periods. It tracks the velocity of price changes relative to a specific past baseline.

In [22]:
df['Momentum_5'] = (
    df['Close'] / df['Close'].shift(5)
) - 1

df['Momentum_10'] = (
    df['Close'] / df['Close'].shift(10)
) - 1

df['Momentum_20'] = (
    df['Close'] / df['Close'].shift(20)
) - 1

In [27]:
df['Volume_MA20'] = (
    df['Volume']
    .rolling(20)
    .mean()
)

df['Volume_Ratio'] = (
    df['Volume']
    /
    df['Volume_MA20']
)

In [28]:
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Log_Returns',
       'Is_Outlier', 'SMA_10', 'SMA_20', 'SMA_50', 'Price_vs_SMA10',
       'Price_vs_SMA20', 'Price_vs_SMA50', 'Momentum_5', 'Momentum_10',
       'Momentum_20', 'Log_Returns_Lag_1', 'Volume_Lag_1', 'Log_Returns_Lag_2',
       'Volume_Lag_2', 'Log_Returns_Lag_3', 'Volume_Lag_3', 'Volatility_7d',
       'Volume_MA20', 'Volume_Ratio'],
      dtype='str')

In [33]:
df = df.dropna().copy() # Remove any remaining rows with missing values

In [34]:
print(df[features].isnull().sum())

Log_Returns          0
Log_Returns_Lag_1    0
Log_Returns_Lag_2    0
Log_Returns_Lag_3    0
Volatility_7d        0
Momentum_5           0
Momentum_10          0
Momentum_20          0
Price_vs_SMA10       0
Price_vs_SMA20       0
Price_vs_SMA50       0
Volume_Ratio         0
dtype: int64


In [36]:
from sklearn.preprocessing import RobustScaler

features = features = [
    'Log_Returns',
    'Log_Returns_Lag_1',
    'Log_Returns_Lag_2',
    'Log_Returns_Lag_3',

    'Volatility_7d',

    'Momentum_5',
    'Momentum_10',
    'Momentum_20',

    'Price_vs_SMA10',
    'Price_vs_SMA20',
    'Price_vs_SMA50',

    'Volume_Ratio'
]
X = df[features]

# 2. Time-Series Split (e.g., 80% training, 20% testing)
split_idx = int(len(X) * 0.8)
train_df = X.iloc[:split_idx]
test_df = X.iloc[split_idx:]

# 3. Fit the scaler ONLY on the training data to prevent leakage
scaler = RobustScaler()
scaler.fit(train_df)

# 4. Transform both training and testing sets
X_train_scaled = scaler.transform(train_df)
X_test_scaled = scaler.transform(test_df)

print("Scaling complete. Mean of scaled training data is now:", X_train_scaled.mean(axis=0))

Scaling complete. Mean of scaled training data is now: [-0.01343585 -0.01285057 -0.01296697 -0.0127406   0.1777009   0.04626872
  0.07810507  0.10611681  0.03607252  0.0794341   0.09441925  0.18019551]


### Here we use XGBoost classifier for predicting market direction (Up/Down), 
### Because we are dealing with a time-series dataset, we must ensure our target arrays (y_train, y_test) align perfectly with the chronological split we performed during the scaling step.

### 1. Prepare Targets and Align the SplitFirst, map out the labels and create the exact same train/test split index used for your scaled features.

In [37]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Create the binary target (1 = Up, 0 = Down/Flat for the NEXT day)
df.loc[:,'Target_Classification'] = (df['Log_Returns'].shift(-1) > 0).astype(int)

# Drop the last row since it has no "tomorrow" to predict
df_clean = df.dropna(subset=['Target_Classification'])

# Re-run your exact split index based on the cleaned data
split_idx = int(len(df_clean) * 0.8)

# Align the target arrays with your scaled features
# (Assuming X_train_scaled and X_test_scaled have been sliced up to split_idx)
y_train = df_clean['Target_Classification'].iloc[:split_idx].values
y_test = df_clean['Target_Classification'].iloc[split_idx:].values

### 2. Train the XGBoost ModelWhen dealing with financial data, standard trees overfit quickly. We add regularization parameters like max_depth (keeping trees shallow) and learning_rate to prevent the model from memorizing noise.

In [38]:
# 2. Initialize the model with conservative parameters for finance
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,            # Shallow trees limit overfitting to market noise
    learning_rate=0.05,     # Small steps yield more stable models
    random_state=42,
    eval_metric='logloss'
)

# 3. Train the model
model.fit(X_train_scaled, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### 3. Evaluate the Results: In crypto trading, raw accuracy can be deceptive. We evaluate using Precision (out of all the times the model predicted "UP", how many times did the price actually go up?) to avoid false buy signals.

In [39]:
# 4. Predict on the test set
y_pred = model.predict(X_test_scaled)

# 5. Print metrics
print("--- Model Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))

--- Model Performance ---
Accuracy Score: 0.5288

Classification Report:
              precision    recall  f1-score   support

   Down/Flat       0.58      0.25      0.35       351
          Up       0.51      0.82      0.63       343

    accuracy                           0.53       694
   macro avg       0.55      0.53      0.49       694
weighted avg       0.55      0.53      0.49       694



For Bitcoin next-day prediction, anything above 50% can be meaningful, but 52.88% alone is not enough. We need to look at precision, recall, and probability distribution.

From the above recall shows that :
Out of all actual UP days - model catches the 82% of them
Out of all actual DOWN days - model catches the 25% of them
So we can say that the model is 'Biased'

In [41]:
y_pred = model.predict(X_test_scaled)
print(pd.Series(y_pred).value_counts())  # This will show how many predictions were made for each class (0 and 1)

1    544
0    150
Name: count, dtype: int64


the model is heavily favoring the "Up" class.

In [45]:
from sklearn.metrics import roc_auc_score

proba = model.predict_proba(X_test_scaled)

auc = roc_auc_score(
    y_test,
    proba[:,1]
)

print("ROC AUC:", auc)
print("**************************************")

proba = model.predict_proba(X_test_scaled)

print(
    pd.Series(proba[:,1]).describe()
)
print("**************************************")

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print(
    importance.sort_values(
        "Importance",
        ascending=False
    )
)

ROC AUC: 0.5339969931806666
**************************************
count    694.000000
mean       0.552375
std        0.069408
min        0.324179
25%        0.510681
50%        0.551843
75%        0.597920
max        0.785573
dtype: float64
**************************************
              Feature  Importance
8      Price_vs_SMA10    0.112668
9      Price_vs_SMA20    0.106154
11       Volume_Ratio    0.094018
3   Log_Returns_Lag_3    0.092087
0         Log_Returns    0.090431
4       Volatility_7d    0.082501
6         Momentum_10    0.080349
5          Momentum_5    0.077667
7         Momentum_20    0.075612
1   Log_Returns_Lag_1    0.065259
10     Price_vs_SMA50    0.063997
2   Log_Returns_Lag_2    0.059257


Trend features are winning

Your newly added features immediately became the most important.

Price_vs_SMA10
Price_vs_SMA20
Volume_Ratio

are all Round 1 features.

That means:

The model is learning trend and volume behavior better than raw returns.

## 1. Add "Lagged" Features (Crucial)
### Right now, your model only looks at today's return and today's volume to predict tomorrow. It has no concept of a trend. You need to give it a window into the past by adding lags:

In [45]:
# Create lag features for the past 3 days
for lag in [1, 2, 3]:
    df.loc[:,f'Log_Returns_Lag_{lag}'] = df['Log_Returns'].shift(lag)
    df.loc[:,f'Volume_Lag_{lag}'] = df['Volume'].shift(lag)

# Add a volatility metric (7-day rolling standard deviation)
df['Volatility_7d'] = df['Log_Returns'].rolling(window=7).std()

df = df.dropna()

In [58]:
# Added new features for the training data set 
features = ['Log_Returns','Log_Returns_Lag_1','Volume_Lag_1','Log_Returns_Lag_2', 'Volume_Lag_2', 'Log_Returns_Lag_3', 'Volume_Lag_3', 'Volatility_7d']
X = df[features]

y = df['Target_Classification'].values

# 5. Time-Series Chronological Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 6. Fit the Scaler strictly on Training Data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. Median of scaled training data is now:", X_train_scaled.mean(axis=0))

Scaling complete. Median of scaled training data is now: [-0.01450238 -0.0139375   0.37174552 -0.01429922  0.37202253 -0.01463754
  0.37197279  0.17923691]


## 2. Address Class Imbalance:
### Since the model is heavily biased toward predicting "Up", you can force XGBoost to penalize errors on "Down" days more heavily.Calculate the ratio of negative days to positive days in your training set.Add scale_pos_weight to your classifier:

In [59]:
# Calculate balance: total_negative_examples / total_positive_examples
negative_cases = (y_train == 0).sum()

positive_cases = (y_train == 1).sum()

ratio = negative_cases / positive_cases
model = XGBClassifier(
    scale_pos_weight=ratio,  # Forces the model to pay attention to Down days
    max_depth=2,             # Lower depth to reduce overfitting to noise
    learning_rate=0.01,
    n_estimators=100
)

model.fit(X_train_scaled, y_train)

,objective,'binary:logistic'
,use_label_encoder,False
,base_score,0.5
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,1
,colsample_bynode,1
,colsample_bytree,1
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## New Model predictions :

In [60]:
# Evaluate updated predictions
y_pred = model.predict(X_test_scaled)
print(f"New Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))

New Accuracy Score: 0.5142
              precision    recall  f1-score   support

   Down/Flat       0.53      0.47      0.50       358
          Up       0.50      0.56      0.53       344

    accuracy                           0.51       702
   macro avg       0.52      0.52      0.51       702
weighted avg       0.52      0.51      0.51       702



## Feature importance findings:

In [61]:

# 1. Extract importance scores from the trained model
importance_scores = model.feature_importances_

# 2. Pair them with your feature names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importance_scores
}).sort_values(by='Importance', ascending=False)

# 3. Print the sorted rankings
print("--- XGBoost Feature Importance Rankings ---")
print(feature_importance_df.to_string(index=False))

--- XGBoost Feature Importance Rankings ---
          Feature  Importance
Log_Returns_Lag_2    0.163486
     Volume_Lag_1    0.156993
    Volatility_7d    0.139314
      Log_Returns    0.133897
     Volume_Lag_2    0.115276
Log_Returns_Lag_3    0.108124
Log_Returns_Lag_1    0.097113
     Volume_Lag_3    0.085797


# Price Prediction Regression:

In [63]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Drop 'Volume' from your features list as it added 0 value
optimized_features = [
    'Log_Returns', 'Log_Returns_Lag_2', 'Volume_Lag_1', 
    'Volatility_7d', 'Volume_Lag_2', 'Log_Returns_Lag_3', 
    'Log_Returns_Lag_1', 'Volume_Lag_3'
]

X_opt = df[optimized_features]

# 2. Define the REGRESSION target (The actual next-day log return value)
y_reg = df['Log_Returns'].shift(-1).dropna()
X_opt = X_opt.iloc[:-1] # Align X by dropping the last row

# 3. Time-Series Chronological Split (80% Train, 20% Test)
split_idx = int(len(X_opt) * 0.8)
X_train, X_test = X_opt.iloc[:split_idx], X_opt.iloc[split_idx:]
y_train, y_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]

# 4. Scale your optimized features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train the XGBoost Regressor
reg_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.03,
    random_state=42,
    eval_metric='mae' # Mean Absolute Error optimization
)
reg_model.fit(X_train_scaled, y_train)

# 6. Predict and Evaluate
y_pred = reg_model.predict(X_test_scaled)
print("--- Regressor Performance ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.6f}")
print(f"R-squared (R2) Score: {r2_score(y_test, y_pred):.4f}")

--- Regressor Performance ---
Mean Absolute Error (MAE): 0.029583
R-squared (R2) Score: -0.7451


## Your model is heavily overfitting—it is chasing the noise of the training data and making wild, incorrect predictions on the test set.

### Why is the Regressor struggling?
### Lags are too short for value extraction: While a 2-day lag can tell you if the market went up or down, it doesn't give a model enough structural       context to guess if Bitcoin will move by $200 or $5,000.
### XGBoost Regressors default to overfitting: Trees are excellent at structural thresholds, but they are notorious for failing at continuous sequence     extrapolation in financial time series.

In [64]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import RobustScaler

# 1. Prepare your optimized features and regression target
optimized_features = [
    'Log_Returns', 'Log_Returns_Lag_2', 'Volume_Lag_1', 
    'Volatility_7d', 'Volume_Lag_2', 'Log_Returns_Lag_3', 
    'Log_Returns_Lag_1', 'Volume_Lag_3'
]

# X is our feature matrix, y is the actual next-day continuous log return
X_raw = df[optimized_features].values
y_raw = df['Log_Returns'].shift(-1).dropna().values
X_raw = X_raw[:-1] # Align lengths

# 2. Scale the data chronologically
split_idx = int(len(X_raw) * 0.8)
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_raw[:split_idx])
X_test_scaled = scaler.transform(X_raw[split_idx:])
y_train, y_test = y_raw[:split_idx], y_raw[split_idx:]

# 3. Create the 3D Sliding Window Function
def create_3d_windows(X_data, y_data, lookback=10):
    X_3d, y_3d = [], []
    for i in range(len(X_data) - lookback):
        X_3d.append(X_data[i : i + lookback]) # Take rows from i to i+lookback
        y_3d.append(y_data[i + lookback])     # Predict the target at the end of the window
    return np.array(X_3d), np.array(y_3d)

# Convert arrays into 3D shapes (Samples, 10 Time Steps, 8 Features)
LOOKBACK_DAYS = 10
X_train_3d, y_train_3d = create_3d_windows(X_train_scaled, y_train, lookback=LOOKBACK_DAYS)
X_test_3d, y_test_3d = create_3d_windows(X_test_scaled, y_test, lookback=LOOKBACK_DAYS)

# 4. Build the LSTM Architecture
model_lstm = Sequential([
    # First LSTM layer requires input_shape = (time_steps, features)
    LSTM(units=50, return_sequences=False, input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
    Dropout(0.2), # Dropout layer prevents overfitting to noise
    Dense(units=25, activation='relu'),
    Dense(units=1) # Single output node for continuous price return forecasting
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train the Network
history = model_lstm.fit(
    X_train_3d, y_train_3d, 
    epochs=20, 
    batch_size=32, 
    validation_data=(X_test_3d, y_test_3d),
    verbose=1
)


C:\Users\Karnulu Suresh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0047 - val_loss: 8.3815e-04
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0022 - val_loss: 7.5443e-04
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0019 - val_loss: 7.6674e-04
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0018 - val_loss: 7.2260e-04
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0017 - val_loss: 7.3818e-04
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.2591e-04
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016 - val_loss: 7.2606e-04
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.1675e-04
Epoch 9/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.1622e-04
Epoch 10/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016 - val_loss: 7.1394e-04
Epoch 11/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0015 - val_loss: 7.1551e-04
Epoch 12/20
88/88 ━━━━━━━

In [65]:
y_pred_lstm = model_lstm.predict(X_test_3d)

print(f"LSTM MAE: {mean_absolute_error(y_test_3d, y_pred_lstm):.6f}")
print(f"LSTM R2 Score: {r2_score(y_test_3d, y_pred_lstm):.4f}")

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
LSTM MAE: 0.017945
LSTM R2 Score: 0.0028


In [70]:
import os
import pickle

# 1. Create directory if missing
os.makedirs('models', exist_ok=True)

# 2. Save RobustScaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ RobustScaler saved to models/scaler.pkl")

# 3. Save XGBoost Classifier using PICKLE (Bypasses Python 3.13 JSON bug)
# (Assuming your variable name is 'model' from the training script)
with open('models/classifier.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✅ XGBoost Classifier saved safely to models/classifier.pkl")

# 4. Save Keras LSTM Regressor
# (Assuming your variable name is 'model_lstm')
model_lstm.save('models/regressor_lstm.h5')
print("✅ LSTM Regressor saved to models/regressor_lstm.h5")


✅ RobustScaler saved to models/scaler.pkl
✅ XGBoost Classifier saved safely to models/classifier.pkl
✅ LSTM Regressor saved to models/regressor_lstm.h5
